In [1]:
import torch
import torchvision
import matplotlib.pyplot
import torch.nn as nn

In [45]:
IMG_DIR = '/home/vedh/workspaces/mlcv_ws/src/datasets/cifar-10/train'
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# ARCHITECTURE PARAMETERS
batch_size = 64
num_classes = 10
num_channels = 3
img_size = 32
patch_size = 8
patch_num = (img_size // patch_size) ** 2
attention_heads = 4
embedding_dim = 32
transformer_blocks = 4
mlp_dim = 64
learning_rate = 0.01
epochs = 10

In [3]:
# Convert the CIFAR-10 dataset into a PyTorch tensor and save it as a .pt file
import numpy as np
import pandas as pd
import torch
from PIL import Image

CSV = "/home/vedh/workspaces/mlcv_ws/src/datasets/cifar-10/trainLabels.csv"

CLASS_NAMES = sorted(["airplane", "automobile", "bird", "cat", "deer",
                      "dog", "frog", "horse", "ship", "truck"])
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

df = pd.read_csv(CSV)
X = np.stack([np.array(Image.open(f"{IMG_DIR}/{i}.png").convert("RGB")) for i in df["id"]])  # (N,32,32,3) uint8
y = df["label"].map(class_to_idx).to_numpy()

X = torch.from_numpy(X).permute(0, 3, 1, 2).contiguous()   # (N,3,32,32) channels-first, done once
y = torch.from_numpy(y).long()

torch.save({"X": X, "y": y, "classes": CLASS_NAMES}, "cifar10_train.pt")
print(X.shape, X.dtype, y.shape)

/tmp/ipykernel_29156/220221106.py:18: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  y = torch.from_numpy(y).long()


torch.Size([50000, 3, 32, 32]) torch.uint8 torch.Size([50000])


In [4]:
# SPLITTING DATASET INTO TRAIN AND VALIDATION SETS ( STRATIFIED SAMPLING )
import numpy as np
from torch.utils.data import Subset, DataLoader, TensorDataset

data = torch.load("cifar10_train.pt")
X, y = data["X"], data["y"]

full = TensorDataset(X, y)

rng = np.random.default_rng(42)
y_np = y.numpy()

train_idx, val_idx = [], []
for c in range(10):
    idx = np.where(y_np == c)[0]
    rng.shuffle(idx)
    val_idx.extend(idx[:1000])
    train_idx.extend(idx[1000:])

train_ds = Subset(full, train_idx)
val_ds   = Subset(full, val_idx)
print(len(train_ds), len(val_ds))     # 40000 10000

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, pin_memory=True)
print(len(train_loader), len(val_loader))  # 625 157

40000 10000
625 157


In [5]:
# SPLIT CHECK
assert len(set(train_idx) & set(val_idx)) == 0          # no overlap
print(np.bincount(y_np[train_idx]))
print(np.bincount(y_np[val_idx]))

[4000 4000 4000 4000 4000 4000 4000 4000 4000 4000]
[1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]


In [6]:
# import tensorflow as tf

# def read_and_decode(filename):
#     ## READ THE RAW FILE
#     raw_bytes = tf.io.read_file(filename)
#     ## DECODE THE IMAGE
#     img = tf.image.decode_png(raw_bytes, channels=num_channels)
#     ## CONVERT PIXEL VALUES TO FLOAT32 in [0, 1]
#     img = tf.image.convert_image_dtype(img, tf.float32)
#     return img

# def parse_csvline(csv_line):
#     id_str, label_string = tf.io.decode_csv(csv_line, record_defaults =["", ""])
#     filename = IMG_DIR + "/" + id_str + ".png"
#     img = read_and_decode(filename)
#     label = tf.argmax(tf.cast(tf.math.equal(CLASS_NAMES, label_string), tf.int32))
#     return img, label

# train_dataset = (tf.data.TextLineDataset("/home/vedh/workspaces/mlcv_ws/src/datasets/cifar-10/trainLabels.csv")
#                  .skip(1)
#                  .shuffle(50000)
#                  .map(parse_csvline, num_parallel_calls=tf.data.AUTOTUNE))


# val_dataset = (train_dataset.skip(40000)
#                 .batch(64)
#                 .prefetch(tf.data.AUTOTUNE)
#                 )

# train_dataset = (
#     train_dataset
#     .take(40000)
#     .batch(64)
#     .prefetch(tf.data.AUTOTUNE)
# )
# print("Train dataset size:", sum(1 for _ in train_dataset))
# print("Validation dataset size:", sum(1 for _ in val_dataset))

In [7]:
# for image_batch, label_batch in train_dataset.take(1):
#     print("Image batch shape:", image_batch.shape)
#     print("Label batch shape:", label_batch.shape)
#     print("Labels:", label_batch.numpy())

In [39]:
# class for PatchEmbedding
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels, embedding_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x)  # (B, D, H', W')
        x = x.flatten(2).permute(0, 2, 1)  # (B, N, D)
        return x

In [33]:
# image_batch, label_batch = next(iter(train_dataset))
# print("Image batch shape:", image_batch.shape)
# x = torch.from_numpy(image_batch.numpy()).permute(0, 3, 1, 2)   # (B, 3, 32, 32)
# print("Resized tensor shape:", x.shape)
# patch_embed = nn.Conv2d(num_channels, embedding_dim, kernel_size=patch_size, stride=patch_size)
# embed_image = patch_embed(x)
# print("Patch embedding shape:", embed_image.shape)
# flattened_embed_image = embed_image.flatten(2).permute(0, 2, 1)   # (B, N, D)
# print("Flattened patch embedding shape:", flattened_embed_image.shape)

In [10]:
# SAME BLOCK AS BEFORE BUT WITH NEW PYTORCH DATASET
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

image_batch, label_batch = next(iter(train_loader))
print("Image batch shape:", image_batch.shape)                  # (B, 3, 32, 32), already channels-first

x = image_batch.to(device).float() / 255.0                      # uint8 -> float32 in [0, 1]

patch_embed = nn.Conv2d(num_channels, embedding_dim,
                        kernel_size=patch_size, stride=patch_size).to(device)
embed_image = patch_embed(x)
print("Patch embedding shape:", embed_image.shape)              # (B, D, H/P, W/P)

flattened_embed_image = embed_image.flatten(2).permute(0, 2, 1)
print("Flattened patch embedding shape:", flattened_embed_image.shape)   # (B, N, D)

Image batch shape: torch.Size([64, 3, 32, 32])
Patch embedding shape: torch.Size([64, 32, 4, 4])
Flattened patch embedding shape: torch.Size([64, 16, 32])


In [40]:
# Class for Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.multi_head_attention = nn.MultiheadAttention(embedding_dim, attention_heads, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embedding_dim),
            # nn.GELU(),
            # nn.Linear(embedding_dim),
        )

    def forward(self, x):
        residual1 = x
        x = self.layer_norm1(x)
        x = self.multi_head_attention(x, x, x)[0] + residual1
        residual2 = x
        x = self.layer_norm2(x)
        x = self.mlp(x) + residual2 
        return x

In [41]:
# Class for ( MLP ) Probability Head
class MlpHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.layernorm1 = nn.LayerNorm(embedding_dim)
        self.mlp_head = nn.Sequential(
            nn.Linear(embedding_dim, num_classes),
            # nn.Softmax(),
        )

    def forward(self, x):
        # x = x[:, 0]
        x = self.layernorm1(x)
        x = self.mlp_head(x)
        return x


In [42]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        self.position_embedding = nn.Parameter(torch.randn(1, patch_num + 1, embedding_dim))
        self.transformer_blocks = nn.Sequential(*[TransformerBlock() for _ in range (transformer_blocks)])
        self.mlp_head = MlpHead()

    def forward(self, x):
        x = self.patch_embedding(x)
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), 1)
        x = x + self.position_embedding
        x = self.transformer_blocks(x)
        x = x[:, 0]
        x = self.mlp_head(x)
        return x

In [46]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

criterion = nn.CrossEntropyLoss()

In [47]:
for i in range(epochs):
    train_loss = 0
    train_acc = 0
    model.train()
    total_images = 0
    correct_images = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        images = images.to(device).float() / 255.0
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        correct_pred = (preds == labels).sum().item()
        correct_images = correct_images + correct_pred
        total_images = total_images + preds.size(0)


    print((correct_images/total_images)*100)

28.939999999999998
40.497499999999995
44.6525
46.6725
48.6875
49.72
51.075
51.692499999999995
52.93750000000001
53.410000000000004
